# Homework: Deep Q-Network and Improvements

This notebook is designed for independent study of the material from the note [`note_10_deep_q_network.md`](../../notes/md/note_10_deep_q_network.md).
You will implement the key elements of a Deep Q-Network for the **LunarLander-v2** environment, and then reinforce the Double and Dueling improvements.

## Learning objectives
- implement a basic DQN with a replay buffer and target network for the `LunarLander-v2` environment
- compare two epsilon-greedy policy schedules: linear and exponential
- implement Double DQN and the Dueling architecture and analyze their effect on training
- formulate observations and recommendations for tuning DQN

## How to do this work
- Go through the notebook top to bottom; close each TODO block with your own code.
- If you are working in Google Colab, first run the dependency installation (see the next block).
- Save intermediate results and screenshots of the plots for the report.
- Fill in the conclusions section at the end of the notebook.

## Useful tips

### About the LunarLander-v2 environment:
- **Task**: land the lunar module between two flags
- **Observations**: 8 continuous values (position, velocity, angle, ground contact)
- **Actions**: 4 discrete actions (do nothing, left engine, main engine, right engine)
- **Rewards**: +100 for a successful landing, penalties for using the engines and for crashing
- **Solved**: average reward ≥ 200 over 100 consecutive episodes
- **Difficulty**: harder than CartPole, requires ~50k-100k steps to train

### Recommended workflow:
1. First make sure the ReplayBuffer works correctly (already implemented)
2. Check the epsilon schedules on the plot
3. Implement and test the loss functions
4. Run the basic DQN on a small number of steps (10k) for debugging
5. Only after that run the full experiments (50k+ steps)

### Notes on DQN for LunarLander:
- **Warmup phase**: the first 5k steps just fill the buffer, no training happens
- **Target network**: updated less often than the policy network (this is key to stability)
- **Epsilon decay**: should finish at around 70-80% of total_steps
- **Batch size**: usually 64-128 for LunarLander
- **Buffer**: at least 50k transitions for stable training

### Environment setup
Run the cell below only if needed (e.g., in Colab).

In [ ]:
# If you're running the notebook in Colab, uncomment the lines below.
# !pip install gymnasium[box2d] torch torchvision matplotlib tqdm -q
# Note: LunarLander requires box2d to be installed (for physics)

In [1]:
import math
import random
from collections import deque, namedtuple
from dataclasses import dataclass
from typing import Callable, Deque, Dict, Iterable, List, Optional, Tuple

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 2024
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

### Getting familiar with LunarLander-v2

Before starting, it's useful to observe the environment:

In [ ]:
# Exploring the LunarLander-v2 environment
env = gym.make("LunarLander-v2")

print("=" * 60)
print("LunarLander-v2 Environment Info")
print("=" * 60)

print(f"\nObservation Space: {env.observation_space}")
print(f"  Shape: {env.observation_space.shape}")
print(f"  Low: {env.observation_space.low}")
print(f"  High: {env.observation_space.high}")

print(f"\nAction Space: {env.action_space}")
print(f"  Number of actions: {env.action_space.n}")
print("\nAction meanings:")
print("  0 - do nothing")
print("  1 - fire left orientation engine")
print("  2 - fire main engine")
print("  3 - fire right orientation engine")

# Let's run one random episode as an example
state, info = env.reset(seed=SEED)
print(f"\nInitial state: {state}")
print("  [0-1]: x, y position")
print("  [2-3]: x, y velocity")
print("  [4]: angle")
print("  [5]: angular velocity")
print("  [6-7]: left leg contact, right leg contact")

total_reward = 0
steps = 0
done = False

print("\nRunning random policy for one episode...")
while not done and steps < 1000:
    action = env.action_space.sample()
    state, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    steps += 1
    done = terminated or truncated

print(f"\nRandom policy results:")
print(f"  Total reward: {total_reward:.2f}")
print(f"  Episode length: {steps} steps")
print(f"  Terminated: {terminated}")
print(f"  Final state: {state}")

env.close()

print("\n" + "=" * 60)
print("Note: LunarLander is considered solved when average reward ≥ 200")
print("=" * 60)

## 1. Replay buffer and transitions
Implement a replay buffer that stores the last `capacity` transitions and is able to:

1. add a new transition, overwriting old ones;
2. sample a random minibatch and return PyTorch tensors.

Use the named tuple `Transition` and store `done` as a float (0.0 or 1.0).

In [2]:
class ReplayBuffer:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, terminated):
        """
        TODO: Store the transition in the buffer.
        Note: we store terminated (a true end of episode),
        not done (which can also include truncation by time limit).
        This matters for computing the TD target correctly.
        """
        # TODO: implement storing
        pass

    def sample(self, batch_size: int) -> Transition:
        """
        TODO: Return a batch of data.
        1. Take a random sample from self.buffer
        2. Unpack it into separate lists/arrays for state, action, reward, ...
        3. Convert to torch.Tensor (on device)
        """
        # TODO: implement sampling
        raise NotImplementedError("TODO: implement sample")

    def __len__(self):
        return len(self.buffer)

### Testing the ReplayBuffer

Let's check that the buffer works correctly:

In [ ]:
# Test ReplayBuffer
test_buffer = ReplayBuffer(capacity=100)

# Add a few transitions
for i in range(5):
    transition = Transition(
        state=np.array([i, i+1, i+2, i+3], dtype=np.float32),
        action=i % 2,
        reward=float(i * 0.1),
        next_state=np.array([i+1, i+2, i+3, i+4], dtype=np.float32),
        done=float(i == 4),
    )
    test_buffer.push(transition)

print(f"Buffer size: {len(test_buffer)}")

# Sample a batch
if len(test_buffer) >= 3:
    batch = test_buffer.sample(batch_size=3)
    print(f"States shape: {batch.state.shape}, device: {batch.state.device}")
    print(f"Actions shape: {batch.action.shape}")
    print(f"Rewards: {batch.reward}")
    print(f"Dones: {batch.done}")
    print("✅ ReplayBuffer works correctly!")
else:
    print("⚠️ Not enough transitions to sample")

## 2. Epsilon-greedy policy and schedules
First implement two epsilon schedule functions: linear and exponential. Then write the action selection strategy.

- The linear schedule should decrease epsilon from `start` to `end` over `duration` steps.
- Exponential schedule: epsilon(t) = epsilon_min + (epsilon_max - epsilon_min) * exp(-t / tau).
- The action selection function returns an int action index.

In [3]:
class LinearSchedule:
    def __init__(self, start: float, end: float, duration: int):
        self.start = start
        self.end = end
        self.duration = duration

    def __call__(self, t: int) -> float:
        """
        TODO: Compute epsilon for step t.
        Linear interpolation from start to end over duration steps.
        After duration steps, return end.
        """
        # TODO: implement the linear schedule
        raise NotImplementedError("TODO: implement LinearSchedule")

### Checking the epsilon schedules

Visualize the schedules before using them:

In [ ]:
# Checking the epsilon schedules
import matplotlib.pyplot as plt

steps = np.arange(0, 20000)

# Linear schedule
linear_schedule = LinearSchedule(start=1.0, end=0.01, duration=15000)
linear_epsilons = [linear_schedule(s) for s in steps]

# Exponential schedule
exp_schedule = ExponentialSchedule(start=1.0, end=0.01, tau=5000)
exp_epsilons = [exp_schedule(s) for s in steps]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(steps, linear_epsilons, label='Linear')
plt.xlabel('Step')
plt.ylabel('Epsilon')
plt.title('Linear Schedule')
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(steps, exp_epsilons, label='Exponential', color='orange')
plt.xlabel('Step')
plt.ylabel('Epsilon')
plt.title('Exponential Schedule')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

print(f"Linear: epsilon at step 0={linear_schedule(0):.3f}, step 15000={linear_schedule(15000):.3f}")
print(f"Exponential: epsilon at step 0={exp_schedule(0):.3f}, step 15000={exp_schedule(15000):.3f}")

## 3. Q-network and target network
For the `CartPole-v1` environment an MLP is enough. Implement the network's forward pass and Xavier weight initialization.

After implementing it, create a separate target network and its update function.

In [4]:
class DQN(nn.Module):
    def __init__(self, observation_dim: int, action_dim: int, hidden_dims: Tuple[int, ...] = (128, 128)):
        super().__init__()
        # TODO: Build the MLP network
        # 1. Input layer: observation_dim -> hidden_dims[0]
        # 2. Hidden layers: hidden_dims[i] -> hidden_dims[i+1] with ReLU
        # 3. Output layer: hidden_dims[-1] -> action_dim (linear, no activation)
        # 4. Weight initialization (e.g., Xavier)

        self.net = nn.Sequential(
            # ...
        )
        raise NotImplementedError("TODO: implement DQN")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: forward pass
        # Don't forget to handle dimensions (if x is one-dimensional)
        pass

### Testing the DQN network

Let's check the output shapes and initialization:

In [ ]:
# Test DQN network
test_env = gym.make("LunarLander-v2")
obs_dim = test_env.observation_space.shape[0]
act_dim = test_env.action_space.n
test_env.close()

print(f"LunarLander-v2:")
print(f"  Observation dim: {obs_dim}")
print(f"  Action dim: {act_dim}")
print(f"  Observation space: {test_env.observation_space}")
print(f"  Action meanings: 0=nothing, 1=fire left, 2=fire main, 3=fire right")

# Create the network
test_dqn = DQN(obs_dim, act_dim, hidden_dims=(128, 128)).to(device)
print(f"\nNetwork architecture:\n{test_dqn}")

# Test forward pass
test_state = torch.randn(1, obs_dim, device=device)
with torch.no_grad():
    q_values = test_dqn(test_state)

print(f"\nInput: {test_state.shape}")
print(f"Output (Q-values): {q_values.shape}")
print(f"Q-values for 4 actions: {q_values}")

# Check action selection
best_action = torch.argmax(q_values, dim=1).item()
print(f"\nBest action: {best_action}")

# Check batch
batch_states = torch.randn(32, obs_dim, device=device)
with torch.no_grad():
    batch_q_values = test_dqn(batch_states)
print(f"\nBatch input: {batch_states.shape}")
print(f"Batch output: {batch_q_values.shape}")

print("\n✅ DQN network works correctly!")

In [5]:
def hard_update(target: nn.Module, source: nn.Module) -> None:
    '''Copies weights from source to target (hard update).'''
    target.load_state_dict(source.state_dict())


def soft_update(target: nn.Module, source: nn.Module, tau: float) -> None:
    '''Smoothly updates the target network with tau between 0 and 1.'''
    with torch.no_grad():
        for target_param, param in zip(target.parameters(), source.parameters()):
            target_param.data.mul_(1 - tau).add_(tau * param.data)

## 4. DQN loss and the training loop
Fill in the functions below. They are responsible for computing the TD error and the main training loop.

1. `compute_td_target` computes r + gamma * max_a Q_target(s_next, a).
2. `compute_dqn_loss` returns the SmoothL1Loss between Q(s,a) and the target.
3. `train_dqn` collects experience, updates the buffer, takes optimizer steps, and logs metrics.

For logging, use a dictionary of lists (`Dict[str, List[float]]`).

In [6]:
@torch.no_grad()
def compute_td_target(
    target_net: nn.Module,
    next_states: torch.Tensor,
    rewards: torch.Tensor,
    dones: torch.Tensor,
    gamma: float
) -> torch.Tensor:
    """
    TODO: Compute the TD target using the formula:
    y_i = r_i + gamma * max_a Q_target(s_{i+1}, a) * (1 - done_i)

    Note:
    - target_net returns Q-values for all actions
    - you need to take the max over actions (dim=1)
    - if done_i = 1, then target = r_i (there is no future)
    """
    # TODO: implement target computation
    raise NotImplementedError("TODO: implement compute_td_target")

## 5. Policy evaluation
Implement deterministic evaluation for the resulting policy (without epsilon randomization).

In [7]:
@torch.no_grad()
def evaluate_policy(env: gym.Env, policy_net: nn.Module, episodes: int = 5) -> float:
    '''Returns the average reward over the given number of episodes for the greedy policy.'''
    policy_net.eval()
    rewards = []
    for _ in range(episodes):
        state, _ = env.reset()
        episode_reward = 0.0
        done = False
        while not done:
            state_tensor = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
            q_values = policy_net(state_tensor)
            action = int(torch.argmax(q_values, dim=1).item())
            next_state, reward, terminated, truncated, _ = env.step(action)
            episode_reward += float(reward)
            state = next_state
            done = terminated or truncated
        rewards.append(episode_reward)
    policy_net.train()
    return float(np.mean(rewards))

## 6. Experiment A - comparing schedules
1. Create two copies of the environment and networks with identical initial weights.
2. Train a DQN with the linear and exponential schedules (20k-40k steps each).
3. Plot `episode_reward` and `epsilon` (use `matplotlib`).
4. Fill in the text conclusions below.

Fill in the code and markdown cells marked TODO.

In [ ]:
# TODO: set up hyperparameters, environments, and schedules
# Hints for LunarLander-v2:
# 1. Create a LunarLander-v2 environment
# 2. Determine the dimensions: observation_dim=8 and action_dim=4
# 3. Create two pairs of networks (policy + target) with identical initial weights
# 4. Define hyperparameters:
#    - total_steps: 80000-120000 (LunarLander needs more steps than CartPole)
#    - warmup_steps: 5000-10000
#    - batch_size: 64-128
#    - gamma: 0.99
#    - learning_rate: 5e-4 (smaller than for CartPole)
#    - buffer_capacity: 50000-100000 (larger for stability)
#    - update_target_every: 1000-2000

# Example structure:
# env_name = "LunarLander-v2"
# env_linear = gym.make(env_name)
# env_exp = gym.make(env_name)
#
# observation_dim = env_linear.observation_space.shape[0]  # 8
# action_dim = env_linear.action_space.n  # 4
#
# # Hyperparameters for LunarLander
# total_steps = 100000
# warmup_steps = 5000
# batch_size = 128
# gamma = 0.99
# lr = 5e-4
# buffer_capacity = 50000
# update_target_every = 1000
#
# # Networks for the linear schedule
# policy_net_linear = DQN(observation_dim, action_dim, hidden_dims=(256, 256)).to(device)
# target_net_linear = DQN(observation_dim, action_dim, hidden_dims=(256, 256)).to(device)
# target_net_linear.load_state_dict(policy_net_linear.state_dict())
#
# # Networks for the exponential schedule (with the same initial weights)
# policy_net_exp = DQN(observation_dim, action_dim, hidden_dims=(256, 256)).to(device)
# policy_net_exp.load_state_dict(policy_net_linear.state_dict())
# target_net_exp = DQN(observation_dim, action_dim, hidden_dims=(256, 256)).to(device)
# target_net_exp.load_state_dict(policy_net_linear.state_dict())
#
# # Optimizers
# optimizer_linear = optim.Adam(policy_net_linear.parameters(), lr=lr)
# optimizer_exp = optim.Adam(policy_net_exp.parameters(), lr=lr)
#
# # Buffers
# buffer_linear = ReplayBuffer(buffer_capacity)
# buffer_exp = ReplayBuffer(buffer_capacity)
#
# # Schedules
# schedule_linear = LinearSchedule(start=1.0, end=0.01, duration=int(0.7 * total_steps))
# schedule_exp = ExponentialSchedule(start=1.0, end=0.01, tau=total_steps / 5)

raise NotImplementedError("TODO: prepare environments, networks, and schedules for experiment A")

NotImplementedError: TODO: prepare environments, networks, and schedules for experiment A

In [ ]:
# TODO: run training for the linear schedule
# Hints:
# 1. Use the train_dqn() function with the linear networks and schedule
# 2. Save the result into the metrics_linear variable
# 3. Evaluate the final policy via evaluate_policy()

# Example:
# print("Training with linear epsilon schedule...")
# metrics_linear = train_dqn(
#     env=env_linear,
#     policy_net=policy_net_linear,
#     target_net=target_net_linear,
#     optimizer=optimizer_linear,
#     buffer=buffer_linear,
#     schedule=schedule_linear,
#     total_steps=total_steps,
#     warmup_steps=warmup_steps,
#     batch_size=batch_size,
#     gamma=gamma,
#     update_target_every=update_target_every,
#     double_dqn=False,
# )
#
# eval_reward_linear = evaluate_policy(env_linear, policy_net_linear, episodes=10)
# print(f"Linear schedule: average evaluation reward = {eval_reward_linear:.1f}")
# env_linear.close()

raise NotImplementedError("TODO: run train_dqn with the linear schedule")

In [ ]:
# TODO: run training for the exponential schedule
# Hint: similar to the previous cell, but with the exponential objects

# Example:
# print("Training with exponential epsilon schedule...")
# metrics_exp = train_dqn(
#     env=env_exp,
#     policy_net=policy_net_exp,
#     target_net=target_net_exp,
#     optimizer=optimizer_exp,
#     buffer=buffer_exp,
#     schedule=schedule_exp,
#     total_steps=total_steps,
#     warmup_steps=warmup_steps,
#     batch_size=batch_size,
#     gamma=gamma,
#     update_target_every=update_target_every,
#     double_dqn=False,
# )
#
# eval_reward_exp = evaluate_policy(env_exp, policy_net_exp, episodes=10)
# print(f"Exponential schedule: average evaluation reward = {eval_reward_exp:.1f}")
# env_exp.close()

raise NotImplementedError("TODO: run train_dqn with the exponential schedule")

In [ ]:
# TODO: visualize the comparison of rewards and epsilon
# Hints:
# 1. Create a figure with 3 plots: episode_reward, epsilon, loss
# 2. Use a moving average to smooth the rewards
# 3. Add a legend with the final evaluation scores

# Helper function for the moving average
def moving_average(values: List[float], window: int = 20) -> np.ndarray:
    arr = np.asarray(values, dtype=np.float32)
    if arr.size < window:
        return arr
    kernel = np.ones(window) / window
    return np.convolve(arr, kernel, mode='valid')

# Visualization example:
# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
#
# # Plot 1: Rewards
# ma_linear = moving_average(metrics_linear['episode_reward'], window=20)
# ma_exp = moving_average(metrics_exp['episode_reward'], window=20)
# axes[0].plot(ma_linear, label=f'Linear (eval={eval_reward_linear:.0f})')
# axes[0].plot(ma_exp, label=f'Exponential (eval={eval_reward_exp:.0f})')
# axes[0].set_title('Episode Rewards (MA 20)')
# axes[0].set_xlabel('Episode')
# axes[0].set_ylabel('Reward')
# axes[0].legend()
# axes[0].grid(True)
#
# # Plot 2: Epsilon
# axes[1].plot(metrics_linear['epsilon'], label='Linear', alpha=0.7)
# axes[1].plot(metrics_exp['epsilon'], label='Exponential', alpha=0.7)
# axes[1].set_title('Epsilon Schedule')
# axes[1].set_xlabel('Episode')
# axes[1].set_ylabel('Epsilon')
# axes[1].legend()
# axes[1].grid(True)
#
# # Plot 3: Loss
# if len(metrics_linear['loss']) > 0:
#     ma_loss_linear = moving_average(metrics_linear['loss'], window=100)
#     ma_loss_exp = moving_average(metrics_exp['loss'], window=100)
#     axes[2].plot(ma_loss_linear, label='Linear')
#     axes[2].plot(ma_loss_exp, label='Exponential')
#     axes[2].set_title('Training Loss (MA 100)')
#     axes[2].set_xlabel('Update Step')
#     axes[2].set_ylabel('Loss')
#     axes[2].legend()
#     axes[2].grid(True)
#
# plt.tight_layout()
# plt.show()

raise NotImplementedError("TODO: plot the results for experiment A")

### Conclusions for experiment A

**Answer the following questions:**

1. **TODO** Which epsilon schedule led to faster reward growth in LunarLander?

2. **TODO** Which scheme showed less fluctuation in rewards? (LunarLander has high reward variance)

3. **TODO** Compare eval_reward for both variants. Did you reach the solved threshold (≥200)?

4. **TODO** Explain how the shape of the epsilon curve affects the exploration/exploitation balance in LunarLander.

5. **TODO** Which schedule would you choose for LunarLander and why?

## 7. Experiment B - Double DQN
Modify the target computation to use the Double DQN formula. Then repeat training (on a short step budget) and compare the TD error distribution.

1. Implement the `compute_double_dqn_target` function below.
2. Modify `compute_dqn_loss` to accept a `double_dqn` flag (create a new function if needed).
3. Rerun training and save the metrics.

In [ ]:
@torch.no_grad()
def compute_double_dqn_target(
    policy_net: nn.Module,
    target_net: nn.Module,
    next_states: torch.Tensor,
    rewards: torch.Tensor,
    dones: torch.Tensor,
    gamma: float,
) -> torch.Tensor:
    '''Computes the Double DQN target, separating action selection and evaluation.'''
    next_actions = policy_net(next_states).argmax(dim=1, keepdim=True)
    next_q = target_net(next_states).gather(1, next_actions).squeeze(1)
    return rewards + gamma * (1.0 - dones) * next_q

In [ ]:
def compute_loss_with_variant(
    policy_net: nn.Module,
    target_net: nn.Module,
    batch: Transition,
    gamma: float,
    mode: str = "vanilla",
) -> torch.Tensor:
    '''Helper for choosing the target computation variant (vanilla or double).'''
    use_double = mode.lower() == "double"
    return compute_dqn_loss(policy_net, target_net, batch, gamma, double_dqn=use_double)

In [ ]:
# TODO: run training with Double DQN and compare metrics
# Hints:
# 1. Create new environments and networks for a fair comparison
# 2. Run two training runs: one with double_dqn=False, the other with double_dqn=True
# 3. Use the same hyperparameters and epsilon schedule
# 4. Compare rewards and training stability
# 5. For LunarLander, Double DQN can significantly reduce Q-value overestimation

# Example structure:
# print("=" * 50)
# print("Experiment B: Vanilla DQN vs Double DQN on LunarLander")
# print("=" * 50)
#
# # Setup for both experiments
# total_steps_b = 80000
# warmup_steps_b = 5000
# batch_size_b = 128
# gamma_b = 0.99
# lr_b = 5e-4
# buffer_capacity_b = 50000
# update_target_every_b = 1000
#
# # Vanilla DQN
# env_vanilla = gym.make("LunarLander-v2")
# obs_dim = env_vanilla.observation_space.shape[0]
# act_dim = env_vanilla.action_space.n
#
# policy_net_vanilla = DQN(obs_dim, act_dim, hidden_dims=(256, 256)).to(device)
# target_net_vanilla = DQN(obs_dim, act_dim, hidden_dims=(256, 256)).to(device)
# target_net_vanilla.load_state_dict(policy_net_vanilla.state_dict())
# optimizer_vanilla = optim.Adam(policy_net_vanilla.parameters(), lr=lr_b)
# buffer_vanilla = ReplayBuffer(buffer_capacity_b)
# schedule_vanilla = LinearSchedule(start=1.0, end=0.01, duration=int(0.7 * total_steps_b))
#
# print("\n1. Training Vanilla DQN...")
# metrics_vanilla = train_dqn(
#     env=env_vanilla,
#     policy_net=policy_net_vanilla,
#     target_net=target_net_vanilla,
#     optimizer=optimizer_vanilla,
#     buffer=buffer_vanilla,
#     schedule=schedule_vanilla,
#     total_steps=total_steps_b,
#     warmup_steps=warmup_steps_b,
#     batch_size=batch_size_b,
#     gamma=gamma_b,
#     update_target_every=update_target_every_b,
#     double_dqn=False,
# )
# eval_vanilla = evaluate_policy(env_vanilla, policy_net_vanilla, episodes=10)
# print(f"Vanilla DQN: eval_reward = {eval_vanilla:.1f}")
# env_vanilla.close()
#
# # Double DQN
# env_double = gym.make("LunarLander-v2")
# policy_net_double = DQN(obs_dim, act_dim, hidden_dims=(256, 256)).to(device)
# policy_net_double.load_state_dict(policy_net_vanilla.state_dict())  # Same initial weights
# target_net_double = DQN(obs_dim, act_dim, hidden_dims=(256, 256)).to(device)
# target_net_double.load_state_dict(policy_net_vanilla.state_dict())
# optimizer_double = optim.Adam(policy_net_double.parameters(), lr=lr_b)
# buffer_double = ReplayBuffer(buffer_capacity_b)
# schedule_double = LinearSchedule(start=1.0, end=0.01, duration=int(0.7 * total_steps_b))
#
# print("\n2. Training Double DQN...")
# metrics_double = train_dqn(
#     env=env_double,
#     policy_net=policy_net_double,
#     target_net=target_net_double,
#     optimizer=optimizer_double,
#     buffer=buffer_double,
#     schedule=schedule_double,
#     total_steps=total_steps_b,
#     warmup_steps=warmup_steps_b,
#     batch_size=batch_size_b,
#     gamma=gamma_b,
#     update_target_every=update_target_every_b,
#     double_dqn=True,
# )
# eval_double = evaluate_policy(env_double, policy_net_double, episodes=10)
# print(f"Double DQN: eval_reward = {eval_double:.1f}")
# env_double.close()
#
# # Comparison visualization
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
#
# ma_vanilla = moving_average(metrics_vanilla['episode_reward'], window=50)
# ma_double = moving_average(metrics_double['episode_reward'], window=50)
#
# axes[0].plot(ma_vanilla, label=f'Vanilla DQN (eval={eval_vanilla:.0f})', alpha=0.8)
# axes[0].plot(ma_double, label=f'Double DQN (eval={eval_double:.0f})', alpha=0.8)
# axes[0].axhline(y=200, color='green', linestyle='--', alpha=0.5, label='Solved threshold')
# axes[0].set_title('LunarLander: Episode Rewards')
# axes[0].set_xlabel('Episode')
# axes[0].set_ylabel('Reward (MA 50)')
# axes[0].legend()
# axes[0].grid(True)
#
# if len(metrics_vanilla['loss']) > 0:
#     ma_loss_vanilla = moving_average(metrics_vanilla['loss'], window=100)
#     ma_loss_double = moving_average(metrics_double['loss'], window=100)
#     axes[1].plot(ma_loss_vanilla, label='Vanilla DQN', alpha=0.8)
#     axes[1].plot(ma_loss_double, label='Double DQN', alpha=0.8)
#     axes[1].set_title('Training Loss (MA 100)')
#     axes[1].set_xlabel('Update Step')
#     axes[1].set_ylabel('Loss')
#     axes[1].legend()
#     axes[1].grid(True)
#
# plt.tight_layout()
# plt.show()

raise NotImplementedError("TODO: compare vanilla DQN and Double DQN")

### Conclusions for experiment B

**Answer the following questions:**

1. **TODO** Did you notice a difference in training stability between Vanilla and Double DQN?

2. **TODO** Which method reached good results faster?

3. **TODO** Compare the eval_reward of both variants.

4. **TODO** How do the loss curves differ between Vanilla and Double DQN?

5. **TODO** In which tasks does Double DQN give the biggest gain?

## 8. Experiment C - Dueling Architecture
Implement the Dueling network and compare training dynamics.

1. Implement the `DuelingDQN` class below.
2. Compare it with the base DQN on the same schedule.
3. Draw a conclusion on whether it provides a speedup or more stability.

In [ ]:
class DuelingDQN(nn.Module):
    def __init__(self, observation_dim: int, action_dim: int, hidden_dims: Tuple[int, ...] = (128, 128)):
        super().__init__()
        dims = (observation_dim,) + hidden_dims
        layers: List[nn.Module] = []
        for in_dim, out_dim in zip(dims[:-1], dims[1:]):
            lin = nn.Linear(in_dim, out_dim)
            init_layer(lin)
            layers.extend([lin, nn.ReLU()])
        self.feature_extractor = nn.Sequential(*layers)
        self.value_head = nn.Linear(hidden_dims[-1], 1)
        self.adv_head = nn.Linear(hidden_dims[-1], action_dim)
        init_layer(self.value_head)
        init_layer(self.adv_head)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Returns Q(s,a) = V(s) + A(s,a) - mean(A(s,a)).'''
        if x.dim() == 1:
            x = x.unsqueeze(0)
        features = self.feature_extractor(x.float())
        value = self.value_head(features)
        advantage = self.adv_head(features)
        advantage = advantage - advantage.mean(dim=1, keepdim=True)
        return value + advantage

In [ ]:
# TODO: train the Dueling DQN and compare results with the baseline
# Hints:
# 1. Use DuelingDQN instead of the regular DQN
# 2. Leave all other parameters the same as in the base DQN
# 3. Compare training speed and stability
# 4. For LunarLander, Dueling can help estimate states better

# Example structure:
# print("=" * 50)
# print("Experiment C: Standard DQN vs Dueling DQN on LunarLander")
# print("=" * 50)
#
# # Hyperparameters
# total_steps_c = 80000
# warmup_steps_c = 5000
# batch_size_c = 128
# gamma_c = 0.99
# lr_c = 5e-4
# buffer_capacity_c = 50000
# update_target_every_c = 1000
#
# # Standard DQN
# env_standard = gym.make("LunarLander-v2")
# obs_dim = env_standard.observation_space.shape[0]
# act_dim = env_standard.action_space.n
#
# policy_net_standard = DQN(obs_dim, act_dim, hidden_dims=(256, 256)).to(device)
# target_net_standard = DQN(obs_dim, act_dim, hidden_dims=(256, 256)).to(device)
# target_net_standard.load_state_dict(policy_net_standard.state_dict())
# optimizer_standard = optim.Adam(policy_net_standard.parameters(), lr=lr_c)
# buffer_standard = ReplayBuffer(buffer_capacity_c)
# schedule_standard = LinearSchedule(start=1.0, end=0.01, duration=int(0.7 * total_steps_c))
#
# print("\n1. Training Standard DQN...")
# metrics_standard = train_dqn(
#     env=env_standard,
#     policy_net=policy_net_standard,
#     target_net=target_net_standard,
#     optimizer=optimizer_standard,
#     buffer=buffer_standard,
#     schedule=schedule_standard,
#     total_steps=total_steps_c,
#     warmup_steps=warmup_steps_c,
#     batch_size=batch_size_c,
#     gamma=gamma_c,
#     update_target_every=update_target_every_c,
#     double_dqn=False,
# )
# eval_standard = evaluate_policy(env_standard, policy_net_standard, episodes=10)
# print(f"Standard DQN: eval_reward = {eval_standard:.1f}")
# env_standard.close()
#
# # Dueling DQN
# env_dueling = gym.make("LunarLander-v2")
# policy_net_dueling = DuelingDQN(obs_dim, act_dim, hidden_dims=(256, 256)).to(device)
# target_net_dueling = DuelingDQN(obs_dim, act_dim, hidden_dims=(256, 256)).to(device)
# target_net_dueling.load_state_dict(policy_net_dueling.state_dict())
# optimizer_dueling = optim.Adam(policy_net_dueling.parameters(), lr=lr_c)
# buffer_dueling = ReplayBuffer(buffer_capacity_c)
# schedule_dueling = LinearSchedule(start=1.0, end=0.01, duration=int(0.7 * total_steps_c))
#
# print("\n2. Training Dueling DQN...")
# metrics_dueling = train_dqn(
#     env=env_dueling,
#     policy_net=policy_net_dueling,
#     target_net=target_net_dueling,
#     optimizer=optimizer_dueling,
#     buffer=buffer_dueling,
#     schedule=schedule_dueling,
#     total_steps=total_steps_c,
#     warmup_steps=warmup_steps_c,
#     batch_size=batch_size_c,
#     gamma=gamma_c,
#     update_target_every=update_target_every_c,
#     double_dqn=False,
# )
# eval_dueling = evaluate_policy(env_dueling, policy_net_dueling, episodes=10)
# print(f"Dueling DQN: eval_reward = {eval_dueling:.1f}")
# env_dueling.close()
#
# # Visualization
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
#
# ma_standard = moving_average(metrics_standard['episode_reward'], window=50)
# ma_dueling = moving_average(metrics_dueling['episode_reward'], window=50)
#
# axes[0].plot(ma_standard, label=f'Standard DQN (eval={eval_standard:.0f})', alpha=0.8)
# axes[0].plot(ma_dueling, label=f'Dueling DQN (eval={eval_dueling:.0f})', alpha=0.8)
# axes[0].axhline(y=200, color='green', linestyle='--', alpha=0.5, label='Solved threshold')
# axes[0].set_title('LunarLander: Standard vs Dueling')
# axes[0].set_xlabel('Episode')
# axes[0].set_ylabel('Reward (MA 50)')
# axes[0].legend()
# axes[0].grid(True)
#
# if len(metrics_standard['loss']) > 0:
#     ma_loss_standard = moving_average(metrics_standard['loss'], window=100)
#     ma_loss_dueling = moving_average(metrics_dueling['loss'], window=100)
#     axes[1].plot(ma_loss_standard, label='Standard DQN', alpha=0.8)
#     axes[1].plot(ma_loss_dueling, label='Dueling DQN', alpha=0.8)
#     axes[1].set_title('Training Loss (MA 100)')
#     axes[1].set_xlabel('Update Step')
#     axes[1].set_ylabel('Loss')
#     axes[1].legend()
#     axes[1].grid(True)
#
# plt.tight_layout()
# plt.show()

raise NotImplementedError("TODO: train the Dueling DQN and compare it with the MLP")

### Conclusions for experiment C

**Answer the following questions:**

1. **TODO** How did the Dueling architecture affect training?

2. **TODO** Does the Dueling DQN learn faster or slower?

3. **TODO** Compare the reward variance between Standard and Dueling.

4. **TODO** Which architecture showed the best eval_reward?

5. **TODO** In which scenarios is the Dueling architecture most useful?

## 9. Self-check questions

1. **TODO** Explain why the replay buffer is critically important for DQN. What happens if you train on consecutive transitions?
2. **TODO** Why is a separate target network needed? Why can't you use a single network for both Q(s,a) and the target?
3. **TODO** When does Double DQN give the biggest gain? Give examples of tasks.
4. **TODO** Explain the intuition behind splitting Q(s,a) = V(s) + A(s,a) in the dueling architecture.
5. **TODO** Which hyperparameters turned out to be the most sensitive in your experiments?
6. **TODO** What signal did you track to determine when to stop training?
7. **TODO** For which tasks is DQN not a good fit? When should you use policy gradient methods instead?